<a href="https://colab.research.google.com/github/25730018/colab-notebooks/blob/main/%E9%80%A3%E7%B6%9A%E7%99%BA%E8%A9%B1%E8%AA%8D%E8%AD%98.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 1. Download dataset and modules

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

MessageError: Error: credential propagation was unsuccessful

In [ ]:
!wget https://github.com/takayama-rado/trado_samples/archive/refs/tags/v0.3.6.zip -O master.zip

In [ ]:
!unzip -o master.zip -d master

In [ ]:
!mv master/trado_samples-0.3.6/src/modules_gislr .

In [ ]:
!rm -rf /content/sample_data

In [ ]:
!ls

# 2. Load library and Dataset

In [ ]:
import glob
import random
import h5py
import copy
import json
import math
import os
import re
import sys
import time
import pandas as pd
from copy import deepcopy
from functools import partial
from inspect import signature
from pathlib import Path
from scipy import signal
from torch.optim.lr_scheduler import ReduceLROnPlateau
# Third party's modules
import numpy as np

import torch

from nltk.metrics.distance import edit_distance

from torch import nn
from torch.nn import functional as F
from torch.utils.data import (
    DataLoader)

from torchvision.transforms import Compose

# Local modules
sys.path.append("modules_gislr")
from modules_gislr.dataset import (
    HDF5Dataset,
    merge_padded_batch)
from modules_gislr.defines import (
    get_fullbody_landmarks
)
from modules_gislr.layers import (
    MultiheadAttention,
    PositionalEncoding,
    PositionwiseFeedForward,
    Identity,
    TransformerEncoder,
    TransformerEncoderLayer,
    apply_norm,
    create_norm
)
from modules_gislr.transforms import (
    InsertTokensForS2S,
    PartsBasedNormalization,
    ReplaceNan,
    SelectLandmarksAndFeature,
    ToTensor
)
from modules_gislr.utils import (
    make_causal_mask,
    make_san_mask,
    select_reluwise_activation
)

In [ ]:
# 1.生データ用の正規化
class NormalizeRawData:
  def __call__(self, data):
      max_val = np.max(np.abs(data))
      if max_val > 0:
        data = data / max_val
      return data

# 2. 生データ用の振幅変換（範囲を指定可能に）
def raw_amplitude_scaling(data, scale_range=[0.7, 1.3]):
    scale_factor = np.random.uniform(*scale_range)
    return data * scale_factor

# 3. 生データ用の時間伸縮（範囲を指定可能にし、returnを追加）
def raw_time_stretch(data, stretch_range=[0.7, 1.3]):
    stretch_factor = np.random.uniform(*stretch_range)
    old_indices = np.arange(len(data))
    new_length = int(len(data) * stretch_factor)

    # 修正：np.linspace を使用
    new_indices = np.linspace(0, len(data) - 1, new_length)
    stretched_data = np.interp(new_indices, old_indices, data)

    return stretched_data # 戻り値を追加

In [ ]:
# ------------------------
# 定数設定 (固定長制限 max_idx を排除)
# ------------------------
min_idx = 0
sf = 4000

nperseg = 128
noverlap = 64
Sn = 60
n_all = 30

# データパス
paths = [
    'drive/MyDrive/suzuki_Data/suzuki/2345文字コマンド/',
    'drive/MyDrive/Dara_2024/hishigae/2024-12-13/emg2024-12-13-13-47-19/',
    'drive/MyDrive/Dara_2024/nasu/2024-12-25/emg2024-12-25-15-39-51/',
    'drive/MyDrive/Dara_2024/sakata/2024-12-25/emg2024-12-25-15-02-47/',
    'drive/MyDrive/Dara_2024/sakataka/2024-12-10/emg2024-12-10-16-33-36'
]

name_hiragana = [
    'あめ', 'あさ', 'いえ', 'いぬ', 'かめ', 'かさ', 'くも', 'くつ', 'すな', 'うし',
    'おはよう', 'ただいま', 'おかえり', 'おやすみ', 'がんばれ', 'もちろん', 'なるほど', 'うれしい', 'おいしい', 'もしもし',
]
name_romaji = [
    'ame', 'asa', 'ie', 'inu', 'kame', 'kasa', 'kumo', 'kutu', 'suna', 'usi',
    'ohayou', 'tadaima', 'okaeri', 'oyasumi', 'ganbare', 'motiron', 'naruhodo', 'uresii', 'oisii', 'mosimosi',
]

dataset_dir = Path("dataset_top10")
dataset_dir.mkdir(parents=True, exist_ok=True)

char_to_index = {ch: idx for idx, ch in enumerate(sorted(list(set("".join(name_hiragana)))))}

with open(dataset_dir / "dictionary.json", "w", encoding="utf-8") as f:
    json.dump(char_to_index, f, indent=4, ensure_ascii=False)

split_hdf5_files = {
    "train": h5py.File(dataset_dir / "0.hdf5", "w"),
    "val": h5py.File(dataset_dir / "1.hdf5", "w"),
    "test": h5py.File(dataset_dir / "2.hdf5", "w"),
}

def process_and_save(file_list, hdf5_file, word_romaji, word_hira, prefix, augment=False):
    sample_count = 0
    label_indices = [char_to_index[ch] for ch in word_hira]

    for file in file_list:
        try:
            df = pd.read_csv(file, skiprows=13, header=None)
            raw1, raw2, raw3 = df.iloc[min_idx:, 0].values, df.iloc[min_idx:, 1].values, df.iloc[min_idx:, 2].values
        except:
            continue

        if augment:
            variations = [
                (raw1, raw2, raw3),
                (raw_amplitude_scaling(raw1, [0.7, 1.0]),
                 raw_amplitude_scaling(raw2, [0.7, 1.0]),
                 raw_amplitude_scaling(raw3, [0.7, 1.0])),
                (raw_amplitude_scaling(raw1, [1.0, 1.3]),
                 raw_amplitude_scaling(raw2, [1.0, 1.3]),
                 raw_amplitude_scaling(raw3, [1.0, 1.3])),
                (raw_time_stretch(raw1, [0.7, 1.0]),
                 raw_time_stretch(raw2, [0.7, 1.0]),
                 raw_time_stretch(raw3, [0.7, 1.0])),
                (raw_time_stretch(raw1, [1.0, 1.3]),
                 raw_time_stretch(raw2, [1.0, 1.3]),
                 raw_time_stretch(raw3, [1.0, 1.3]))
            ]
        else:
            variations = [(raw1, raw2, raw3)]

        for v1, v2, v3 in variations:
            _, _, Sxx1 = signal.spectrogram(v1, sf, nperseg=nperseg, noverlap=noverlap, window='hann')
            _, _, Sxx2 = signal.spectrogram(v2, sf, nperseg=nperseg, noverlap=noverlap, window='hann')
            _, _, Sxx3 = signal.spectrogram(v3, sf, nperseg=nperseg, noverlap=noverlap, window='hann')

            min_t = min(Sxx1.shape[1], Sxx2.shape[1], Sxx3.shape[1])
            Sxx = np.stack([Sxx1[:Sn, :min_t].T, Sxx2[:Sn, :min_t].T, Sxx3[:Sn, :min_t].T], axis=0)
            final_feature = 24 * np.log(np.clip(Sxx, a_min=1e-10, a_max=None))

            s_name = f"{prefix}{word_romaji}_{sample_count}"
            g = hdf5_file.create_group(s_name)
            g.create_dataset("feature", data=final_feature)
            g.create_dataset("token", data=np.array(label_indices, dtype=np.int64))
            sample_count += 1
    return sample_count

n_train, n_val, n_test = 24, 3, 3
for path in paths:
    print(f"🔍 {path}")
    path_prefix = Path(path).name + "_"
    for r_name, h_name in zip(name_romaji, name_hiragana):
        file_list = glob.glob(str(Path(path) / r_name / '*.csv'))
        if not file_list: continue
        random.shuffle(file_list)
        c_train = process_and_save(file_list[:n_train], split_hdf5_files["train"], r_name, h_name, path_prefix, augment=True)
        c_val = process_and_save(file_list[n_train:n_train+n_val], split_hdf5_files["val"], r_name, h_name, path_prefix, augment=False)
        c_test = process_and_save(file_list[n_train+n_val:n_train+n_val+n_test], split_hdf5_files["test"], r_name, h_name, path_prefix, augment=False)
        print(f"✅ {h_name} ({r_name}): Train={c_train}, Val={c_val}, Test={c_test}")

for f in split_hdf5_files.values(): f.close()
print("✨ すべて可変長・5倍拡張でHDF5に保存されました。")

In [ ]:
!cat dataset_top10/dictionary.json

In [ ]:
with h5py.File("dataset_top10/0.hdf5", "r") as fread:
    keys = list(fread.keys())  # グループ名のリスト
    print(keys)  # 複数のグループが表示されるはず
    group = fread[keys[1]]  # 最初のグループにアクセス
    print(group.keys())  # "feature" と "token" のキーが存在
    feature = group["feature"][:]  # 特徴量データを読み込む
    token = group["token"][:]  # ラベルデータを読み込む
    print(feature.shape)
    print(token)

In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-

"""train_functions: Define functions for training.
-------------------------------------------------------------------------------



Copyright (c) 2024 N.Takayama @ TRaD <takayaman@takayama-rado.com>
-------------------------------------------------------------------------------
"""

# Standard modules
from __future__ import absolute_import
from __future__ import division
from __future__ import print_function
from __future__ import unicode_literals

import time
from inspect import signature

# Third party's modules
import numpy as np

import torch
import torch.nn.functional as F
from torch import nn

from nltk.metrics.distance import edit_distance


# Execution settings
VERSION = u"%(prog)s dev"

# Input data directories
DIR_INPUT = None
# Output data directories
DIR_OUTPUT = None


def train_loop(dataloader, model, loss_fn, optimizer, device, use_mask=True,
               return_pred_times=False):
    num_batches = len(dataloader)
    train_loss = 0
    size = len(dataloader.dataset)

    # Inspect model signature.
    sig = signature(model.forward)
    use_mask = True if "feature_pad_mask" in sig.parameters and use_mask is True else False

    # Collect prediction time.
    pred_times = []

    # Switch to training mode.
    model.train()
    # Main loop.
    print("Start training.")
    start = time.perf_counter()
    for batch_idx, batch_sample in enumerate(dataloader):
        feature = batch_sample["feature"]
        token = batch_sample["token"]
        feature = feature.to(device)
        token = token.to(device)
        frames = feature.shape[-2]

        # Predict.
        pred_start = time.perf_counter()
        if use_mask:
            feature_pad_mask = batch_sample["feature_pad_mask"]
            feature_pad_mask = feature_pad_mask.to(device)
            pred = model(feature, feature_pad_mask=feature_pad_mask)
        else:
            pred = model(feature)
        pred_end = time.perf_counter()
        pred_times.append([frames, pred_end - pred_start])

        # Compute loss.
        loss = loss_fn(pred, token.squeeze(-1))

        # Back propagation.
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        train_loss += loss.item()

        # Print current loss per 100 steps.
        if batch_idx % 100 == 0:
            loss = loss.item()
            steps = batch_idx * len(feature)
            print(f"loss:{loss:>7f} [{steps:>5d}/{size:>5d}]")
    print(f"Done. Time:{time.perf_counter()-start}")
    # Average loss.
    train_loss /= num_batches
    print("Training performance: \n",
          f"Avg loss:{train_loss:>8f}\n")
    pred_times = np.array(pred_times)
    retval = (train_loss, pred_times) if return_pred_times else train_loss
    return retval


def val_loop(dataloader, model, loss_fn, device, use_mask=True,
             return_pred_times=False):
    num_batches = len(dataloader)
    val_loss = 0

    # Inspect model signature.
    sig = signature(model.forward)
    use_mask = True if "feature_pad_mask" in sig.parameters and use_mask is True else False

    # Collect prediction time.
    pred_times = []

    # Switch to evaluation mode.
    model.eval()
    # Main loop.
    print("Start validation.")
    start = time.perf_counter()
    with torch.no_grad():
        for batch_sample in dataloader:
            feature = batch_sample["feature"]
            token = batch_sample["token"]
            feature = feature.to(device)
            token = token.to(device)
            frames = feature.shape[-2]

            # Predict.
            pred_start = time.perf_counter()
            if use_mask:
                feature_pad_mask = batch_sample["feature_pad_mask"]
                feature_pad_mask = feature_pad_mask.to(device)
                pred = model(feature, feature_pad_mask=feature_pad_mask)
            else:
                pred = model(feature)
            pred_end = time.perf_counter()
            pred_times.append([frames, pred_end - pred_start])

            val_loss += loss_fn(pred, token.squeeze(-1)).item()
    print(f"Done. Time:{time.perf_counter()-start}")

    # Average loss.
    val_loss /= num_batches
    print("Validation performance: \n",
          f"Avg loss:{val_loss:>8f}\n")
    pred_times = np.array(pred_times)
    retval = (val_loss, pred_times) if return_pred_times else val_loss
    return retval


def test_loop(dataloader, model, device, use_mask=False,
              return_pred_times=False):
    size = len(dataloader.dataset)
    correct = 0

    # Inspect model signature.
    sig = signature(model.forward)
    use_mask = True if "feature_pad_mask" in sig.parameters and use_mask is True else False

    # Collect prediction time.
    pred_times = []

    # Switch to evaluation mode.
    model.eval()
    # Main loop.
    print("Start evaluation.")
    start = time.perf_counter()
    with torch.no_grad():
        for batch_sample in dataloader:
            feature = batch_sample["feature"]
            token = batch_sample["token"]
            feature = feature.to(device)
            token = token.to(device)
            frames = feature.shape[-2]

            # Predict.
            pred_start = time.perf_counter()
            if use_mask:
                feature_pad_mask = batch_sample["feature_pad_mask"]
                feature_pad_mask = feature_pad_mask.to(device)
                pred = model(feature, feature_pad_mask=feature_pad_mask)
            else:
                pred = model(feature)
            pred_end = time.perf_counter()
            pred_times.append([frames, pred_end - pred_start])

            pred_ids = pred.argmax(dim=1).unsqueeze(-1)
            count = (pred_ids == token).sum().detach().cpu().numpy()
            correct += int(count)
    print(f"Done. Time:{time.perf_counter()-start}")

    acc = correct / size * 100
    print("Test performance: \n",
          f"Accuracy:{acc:>0.1f}%")
    pred_times = np.array(pred_times)
    retval = (acc, pred_times) if return_pred_times else acc
    return retval


def forward(model, feature, tokens, feature_pad_mask, tokens_pad_mask,
            tokens_causal_mask=None):
    if isinstance(model, TransformerCsEMG):
        if tokens_causal_mask is None:
            tokens_causal_mask = make_causal_mask(tokens_pad_mask)
        if tokens_causal_mask.shape[-1] != tokens_pad_mask.shape[-1]:
            tokens_causal_mask = make_causal_mask(tokens_pad_mask)
        preds = model(src_feature=feature,
                      tgt_feature=tokens,
                      src_causal_mask=None,
                      src_padding_mask=feature_pad_mask,
                      tgt_causal_mask=tokens_causal_mask,
                      tgt_padding_mask=tokens_pad_mask)
    else:
        raise NotImplementedError(f"Unknown model type:{type(model)}.")
    return preds, tokens_causal_mask


def check_tokens_format(tokens, tokens_pad_mask, start_id, end_id):
    # Check token's format.
    end_indices0 = np.arange(len(tokens))
    end_indices1 = tokens_pad_mask.sum(dim=-1).detach().cpu().numpy() - 1
    message = "The start and/or end ids are not included in tokens. " \
        f"Please check data format. start_id:{start_id}, " \
        f"end_id:{end_id}, enc_indices:{end_indices1}, tokens:{tokens}"
    ref_tokens = tokens.detach().cpu().numpy()
    assert (ref_tokens[:, 0] == start_id).all(), message
    assert (ref_tokens[end_indices0, end_indices1] == end_id).all(), message


def train_loop_csir_s2s(dataloader,
                        model,
                        loss_fn,
                        optimizer,
                        device,
                        start_id,
                        end_id,
                        return_pred_times=False):
    num_batches = len(dataloader)
    train_loss = 0
    size = len(dataloader.dataset)

    # Collect prediction time.
    pred_times = []

    # Switch to training mode.
    model.train()
    # Main loop.
    print("Start training.")
    start = time.perf_counter()
    tokens_causal_mask = None
    for batch_idx, batch_sample in enumerate(dataloader):
        feature = batch_sample["feature"]
        feature_pad_mask = batch_sample["feature_pad_mask"]
        tokens = batch_sample["token"]
        tokens_pad_mask = batch_sample["token_pad_mask"]

        check_tokens_format(tokens, tokens_pad_mask, start_id, end_id)

        feature = feature.to(device)
        feature_pad_mask = feature_pad_mask.to(device)
        tokens = tokens.to(device)
        tokens_pad_mask = tokens_pad_mask.to(device)

        frames = feature.shape[-2]

        # Predict.
        pred_start = time.perf_counter()
        preds, tokens_causal_mask = forward(model, feature, tokens,
                                            feature_pad_mask, tokens_pad_mask,
                                            tokens_causal_mask)
        pred_end = time.perf_counter()
        pred_times.append([frames, pred_end - pred_start])

        # Compute loss.
        # Preds do not include <start>, so skip that of tokens.
        loss = 0
        if isinstance(loss_fn, nn.CrossEntropyLoss):
            # 予測: [Batch, Time, Class] -> [Batch * Time, Class] に平坦化
            preds_flat = preds.reshape(-1, preds.shape[-1])
            # 正解: <start>トークンをスキップした [Batch, Time] -> [Batch * Time] に平坦化
            tokens_flat = tokens[:, 1:].reshape(-1)

            # PyTorchのLoss関数に一括処理させる（自動でignore_indexが適用され、正しい平均が取られます）
            loss = loss_fn(preds_flat, tokens_flat)
        # LabelSmoothingCrossEntropyLoss
        else:
            # `[N, T, C] -> [N, C, T]`
            preds = preds.permute([0, 2, 1])
            # Remove prediction after the last token.
            if preds.shape[-1] == tokens.shape[-1]:
                preds = preds[:, :, :-1]
            loss = loss_fn(preds, tokens[:, 1:])

        # Back propagation.
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        train_loss += loss.item()

        # Print current loss per 100 steps.
        if batch_idx % 100 == 0:
            loss = loss.item()
            steps = batch_idx * len(feature)
            print(f"loss:{loss:>7f} [{steps:>5d}/{size:>5d}]")
    print(f"Done. Time:{time.perf_counter()-start}")
    # Average loss.
    train_loss /= num_batches
    print("Training performance: \n",
          f"Avg loss:{train_loss:>8f}\n")
    pred_times = np.array(pred_times)
    retval = (train_loss, pred_times) if return_pred_times else train_loss
    return retval


def val_loop_csir_s2s(dataloader,
                      model,
                      loss_fn,
                      device,
                      start_id,
                      end_id,
                      return_pred_times=False):
    num_batches = len(dataloader)
    val_loss = 0

    # Collect prediction time.
    pred_times = []

    # Switch to evaluation mode.
    model.eval()
    # Main loop.
    print("Start validation.")
    start = time.perf_counter()
    tokens_causal_mask = None
    with torch.no_grad():
        for batch_idx, batch_sample in enumerate(dataloader):
            feature = batch_sample["feature"]
            feature_pad_mask = batch_sample["feature_pad_mask"]
            tokens = batch_sample["token"]
            tokens_pad_mask = batch_sample["token_pad_mask"]

            check_tokens_format(tokens, tokens_pad_mask, start_id, end_id)

            feature = feature.to(device)
            feature_pad_mask = feature_pad_mask.to(device)
            tokens = tokens.to(device)
            tokens_pad_mask = tokens_pad_mask.to(device)

            frames = feature.shape[-2]

            # Predict.
            pred_start = time.perf_counter()
            preds, tokens_causal_mask = forward(model, feature, tokens,
                                                feature_pad_mask, tokens_pad_mask,
                                                tokens_causal_mask)
            pred_end = time.perf_counter()
            pred_times.append([frames, pred_end - pred_start])

            # Compute loss.
            # Preds do not include <start>, so skip that of tokens.
            loss = 0
            if isinstance(loss_fn, nn.CrossEntropyLoss):
                # 予測: [Batch, Time, Class] -> [Batch * Time, Class] に平坦化
                preds_flat = preds.reshape(-1, preds.shape[-1])
                # 正解: <start>トークンをスキップした [Batch, Time] -> [Batch * Time] に平坦化
                tokens_flat = tokens[:, 1:].reshape(-1)

                # PyTorchのLoss関数に一括処理させる（自動でignore_indexが適用され、正しい平均が取られます）
                loss = loss_fn(preds_flat, tokens_flat)
            # LabelSmoothingCrossEntropyLoss
            else:
                # `[N, T, C] -> [N, C, T]`
                preds = preds.permute([0, 2, 1])
                # Remove prediction after the last token.
                if preds.shape[-1] == tokens.shape[-1]:
                    preds = preds[:, :, :-1]
                loss = loss_fn(preds, tokens[:, 1:])

            val_loss += loss.item()
    print(f"Done. Time:{time.perf_counter()-start}")

    # Average loss.
    val_loss /= num_batches
    print("Validation performance: \n",
          f"Avg loss:{val_loss:>8f}\n")
    pred_times = np.array(pred_times)
    retval = (val_loss, pred_times) if return_pred_times else val_loss
    return retval

def inference(model, feature, start_id, end_id, max_seqlen):
    if isinstance(model, TransformerCsEMG):
        pred_ids, _ = model.inference(feature,
                                      start_id,
                                      end_id,
                                      max_seqlen=max_seqlen)
    else:
        raise NotImplementedError(f"Unknown model type:{type(model)}.")
    return pred_ids

def test_loop_csir_s2s(dataloader,
                       model,
                       device,
                       start_id,
                       end_id,
                       max_seqlen=62,
                       return_pred_times=False,
                       verbose_num=1):
    size = len(dataloader.dataset)
    total_cer = 0
    all_preds = []
    all_refs = []

    # Collect prediction time.
    pred_times = []

    # Switch to evaluation mode.
    model.eval()
    # Main loop.
    print("Start test.")
    start = time.perf_counter()
    with torch.no_grad():
        for batch_idx, batch_sample in enumerate(dataloader):
            feature = batch_sample["feature"]
            tokens = batch_sample["token"]
            tokens_pad_mask = batch_sample["token_pad_mask"]

            check_tokens_format(tokens, tokens_pad_mask, start_id, end_id)

            feature = feature.to(device)
            tokens = tokens.to(device)
            tokens_pad_mask = tokens_pad_mask.to(device)

            frames = feature.shape[-2]

            # Predict.
            pred_start = time.perf_counter()
            pred_ids = inference(model, feature, start_id, end_id, max_seqlen)
            pred_end = time.perf_counter()
            pred_times.append([frames, pred_end - pred_start])

            # Compute cer.
            # <sos> and <eos> should be removed because they may boost performance.
            # print(tokens)
            # print(pred_ids)
            if batch_idx < verbose_num:
                print("="*40)
                print("Verbose output")
            if batch_idx < verbose_num:
                print(f"Tokens_w_keywords: {tokens}")
                print(f"Preds_w_keywords: {pred_ids}")

            tokens = tokens[tokens_pad_mask]
            if len(tokens.shape) == 2:
                tokens = tokens[0, 1:-1]
            else:
                tokens = tokens[1:-1]
            # pred_ids = pred_ids[0, 1:-1]
            pred_ids = [pid for pid in pred_ids[0] if pid not in [start_id, end_id]]
            if batch_idx < verbose_num:
                print(f"Tokens_wo_keywords: {tokens}")
                print(f"Preds_wo_keywords: {pred_ids}")

            ref_length = len(tokens)
            cer = edit_distance(tokens, pred_ids)
            cer /= ref_length
            total_cer += cer
            # pred_ids は list, tokens は torch.Tensor
            # tokens は tensorなので list に変換
            ref_tokens_list = tokens.cpu().tolist()
            all_refs.append(ref_tokens_list)
            all_preds.append(pred_ids)
            if batch_idx < verbose_num:
                print(f"cer: {cer}")
                print("="*40)
    print(f"Done. Time:{time.perf_counter()-start}")

    # Average cer.
    acer = total_cer / size * 100
    print("Test performance: \n",
          f"Avg cer:{acer:>0.1f}%\n")
    pred_times = np.array(pred_times)
    if return_pred_times:
        retval = (acer, pred_times, all_preds, all_refs)
    else:
        retval = (acer, all_preds, all_refs)
    return retval

class LabelSmoothingCrossEntropyLoss(nn.Module):
    """Cross-entropy loss with label smoothing.

    For the detail, please refer
    "Rethinking the Inception Architecture for Computer Vision"
    https://arxiv.org/abs/1512.00567
    """
    def __init__(self, weight=None, ignore_indices=None, reduction="none",
                 label_smoothing=0.0):
        super().__init__()
        self.weight = weight
        if isinstance(ignore_indices, int):
            self.ignore_indices = [ignore_indices]
        else:
            self.ignore_indices = ignore_indices
        assert reduction in ["none",
                             "mean_batch_prior", "mean_temporal_prior",
                             "sum"]
        self.reduction = reduction
        assert label_smoothing >= 0.0
        assert label_smoothing <= 1.0
        self.label_smoothing = label_smoothing

    def _isnotin_ignore(self, target):
        # Please refer
        # https://github.com/pytorch/pytorch/issues/3025
        # pylint error of torch.tensor() should be solved in the future release.
        # https://github.com/pytorch/pytorch/issues/24807
        ignore = torch.tensor(self.ignore_indices, dtype=target.dtype,
                              device=target.device)
        isin = (target[..., None] == ignore).any(-1)
        return isin.bitwise_not()

    def _calc_loss(self, logit_t, target_t):
        logit_mask = torch.ones(logit_t.shape[-1],
                                dtype=logit_t.dtype,
                                device=logit_t.device)
        target_mask = torch.ones(target_t.shape,
                                 dtype=logit_t.dtype,
                                 device=logit_t.device)
        if self.ignore_indices is not None:
            logit_mask[self.ignore_indices] = 0
            target_mask = self._isnotin_ignore(target_t).float()
        if self.weight is None:
            weight = torch.ones(logit_t.shape[-1],
                                dtype=logit_t.dtype,
                                device=logit_t.device)
        else:
            weight = self.weight.to(dtype=logit_t.dtype, device=logit_t.device)
        # Calculate CE.
        logprobs = F.log_softmax(logit_t, dim=-1)
        logprobs_m = logprobs * weight * logit_mask
        nll_loss = -logprobs_m.gather(dim=-1, index=target_t.unsqueeze(1))
        nll_loss = nll_loss.squeeze(1)
        smooth_loss = -logprobs_m.sum(dim=-1) / logit_mask.sum()
        smooth_loss *= target_mask
        loss = (1 - self.label_smoothing) * nll_loss + self.label_smoothing * smooth_loss
        return loss

    def forward(self, logit, target):
        """Perform forward computation.

        # Args:
          - logit: `[N, C]` or `[N, C, T]`
          - target: `[N]` or [N, T]
        """
        # Check format.
        if len(logit.shape) == 2:
            logit = logit.unsqueeze(-1)
        if len(target.shape) == 1:
            target = target.unsqueeze(-1)
        assert len(logit.shape) == 3, f"{logit.shape}"
        assert len(target.shape) == 2, f"{target.shape}"
        assert logit.shape[0] == target.shape[0], f"{logit.shape, target.shape}"
        assert logit.shape[-1] == target.shape[-1], f"{logit.shape, target.shape}"

        loss = 0
        for t in range(target.shape[-1]):
            _loss = self._calc_loss(logit[:, :, t], target[:, t])
            # Reduction should be conducted in a loop when reduction is
            # mean_batch_prior.
            if self.reduction == "mean_batch_prior":
                if self.ignore_indices is not None:
                    denom = len([t for t in target[:, t]
                                 if t not in self.ignore_indices])
                else:
                    denom = logit.shape[0]
                _loss /= max(denom, 1)
            loss += _loss

        # Reduction.
        if self.reduction == "sum":
            loss = loss.sum()
        # Temporal Normalization.
        if self.reduction == "mean_batch_prior":
            loss = loss.sum() / target.shape[-1]
        if self.reduction == "mean_temporal_prior":
            target_lengths = self._isnotin_ignore(target).sum(dim=-1)
            loss /= torch.clamp(target_lengths, min=1)
            loss = loss.mean()
        return loss


# --- Execution --------------------------------------------------------
if __name__ == "__main__":
    print(__doc__)

In [ ]:
# Patch for train_functions.
# This is only required for this script.
def forward(model, feature, tokens, feature_pad_mask, tokens_pad_mask, tokens_causal_mask=None):
    if isinstance(model, TransformerCsEMG):
        if tokens_causal_mask is None:
            tokens_causal_mask = make_causal_mask(tokens_pad_mask)
        if tokens_causal_mask.shape[-1] != tokens_pad_mask.shape[-1]:
            tokens_causal_mask = make_causal_mask(tokens_pad_mask)
        preds = model(src_feature=feature,
                      tgt_feature=tokens,
                      src_causal_mask=None,
                      src_padding_mask=feature_pad_mask,
                      tgt_causal_mask=tokens_causal_mask,
                      tgt_padding_mask=tokens_pad_mask)
    else:
        raise NotImplementedError(f"Unknown model type:{type(model)}.")
    return preds, tokens_causal_mask

def inference(model, feature, start_id, end_id, max_seqlen=20):
    if isinstance(model, TransformerCsEMG):
        pred_ids, _ = model.inference(feature,
                                      start_id,
                                      end_id,
                                      max_seqlen=max_seqlen)
    else:
        raise NotImplementedError(f"Unknown model type:{type(model)}.")
    return pred_ids

from modules_gislr import train_functions
train_functions.forward = forward
train_functions.inference = inference

from modules_gislr.train_functions import (
    LabelSmoothingCrossEntropyLoss,
    train_loop_csir_s2s,
    val_loop_csir_s2s,
    test_loop_csir_s2s)

# 3. Implement Transformer Encoder-Decoder CsEMG model

### Transformer Decoder

In [ ]:
class TransformerDecoderLayer(nn.Module):
    def __init__(self,
                 dim_model,
                 num_heads,
                 dim_ffw,
                 dropout,
                 activation,
                 norm_type_sattn,
                 norm_type_cattn,
                 norm_type_ffw,
                 norm_eps,
                 norm_first,
                 add_bias):
        super().__init__()

        self.norm_first = norm_first

        #################################################
        # MHSA.
        #################################################
        self.self_attn = MultiheadAttention(
            key_dim=dim_model,
            query_dim=dim_model,
            att_dim=dim_model,
            out_dim=dim_model,
            num_heads=num_heads,
            dropout=dropout,
            add_bias=add_bias)
        self.norm_sattn = create_norm(norm_type_sattn, dim_model, norm_eps, add_bias)

        #################################################
        # MHCA.
        #################################################
        self.cross_attn = MultiheadAttention(
            key_dim=dim_model,
            query_dim=dim_model,
            att_dim=dim_model,
            out_dim=dim_model,
            num_heads=num_heads,
            dropout=dropout,
            add_bias=add_bias)
        self.norm_cattn = create_norm(norm_type_cattn, dim_model, norm_eps, add_bias)

        #################################################
        # PFFN.
        #################################################
        self.ffw = PositionwiseFeedForward(
            dim_model=dim_model,
            dim_ffw=dim_ffw,
            dropout=dropout,
            activation=activation,
            add_bias=add_bias)
        self.norm_ffw = create_norm(norm_type_ffw, dim_model, norm_eps, add_bias)

        self.dropout = nn.Dropout(p=dropout)

        # To store attention weights.
        self.sattw = None
        self.cattw = None

    def _forward_prenorm(self,
                         tgt_feature,
                         enc_feature,
                         tgt_san_mask,
                         enc_tgt_mask):
        """Pre-normalization structure.

        For the details, please refer
        https://arxiv.org/pdf/2002.04745v1.pdf
        """
        #################################################
        # self-attention
        #################################################
        residual = tgt_feature
        tgt_feature = apply_norm(self.norm_sattn, tgt_feature)
        tgt_feature, self.sattw = self.self_attn(
            key=tgt_feature,
            value=tgt_feature,
            query=tgt_feature,
            mask=tgt_san_mask)
        tgt_feature = self.dropout(tgt_feature) + residual

        #################################################
        # cross-attention
        #################################################
        residual = tgt_feature
        tgt_feature = apply_norm(self.norm_cattn, tgt_feature)
        tgt_feature, self.cattw = self.cross_attn(
            key=enc_feature,
            value=enc_feature,
            query=tgt_feature,
            mask=enc_tgt_mask)
        tgt_feature = self.dropout(tgt_feature) + residual

        #################################################
        # FFW
        #################################################
        residual = tgt_feature
        tgt_feature = apply_norm(self.norm_ffw, tgt_feature)
        tgt_feature = self.ffw(tgt_feature)
        tgt_feature = self.dropout(tgt_feature) + residual
        return tgt_feature

    def _forward_postnorm(self,
                          tgt_feature,
                          enc_feature,
                          tgt_san_mask,
                          enc_tgt_mask):
        """Post-normalization structure (standard).

        """
        #################################################
        # self-attention
        #################################################
        residual = tgt_feature
        tgt_feature, self.sattw = self.self_attn(
            key=tgt_feature,
            value=tgt_feature,
            query=tgt_feature,
            mask=tgt_san_mask)
        tgt_feature = self.dropout(tgt_feature) + residual
        tgt_feature = apply_norm(self.norm_sattn, tgt_feature)

        #################################################
        # cross-attention
        #################################################
        residual = tgt_feature
        tgt_feature, self.cattw = self.cross_attn(
            key=enc_feature,
            value=enc_feature,
            query=tgt_feature,
            mask=enc_tgt_mask)
        tgt_feature = self.dropout(tgt_feature) + residual
        tgt_feature = apply_norm(self.norm_cattn, tgt_feature)

        #################################################
        # FFW
        #################################################
        residual = tgt_feature
        tgt_feature = self.ffw(tgt_feature)
        tgt_feature = self.dropout(tgt_feature) + residual
        tgt_feature = apply_norm(self.norm_ffw, tgt_feature)

        return tgt_feature

    def forward(self,
                tgt_feature,
                enc_feature,
                tgt_causal_mask=None,
                enc_tgt_causal_mask=None,
                tgt_key_padding_mask=None,
                enc_key_padding_mask=None):

        # Create mask.
        if tgt_key_padding_mask is None:
            tgt_key_padding_mask = torch.ones(tgt_feature.shape[:2],
                                              dtype=enc_feature.dtype,
                                              device=enc_feature.device)
        tgt_san_mask = make_san_mask(tgt_key_padding_mask, tgt_causal_mask)
        if enc_key_padding_mask is None:
            enc_key_padding_mask = torch.ones(enc_feature.shape[:2],
                                              dtype=enc_feature.dtype,
                                              device=enc_feature.device)
        enc_tgt_mask = enc_key_padding_mask.unsqueeze(1).repeat(
            [1, tgt_feature.shape[1], 1])
        if enc_tgt_causal_mask is not None:
            enc_tgt_mask = enc_tgt_mask & enc_tgt_causal_mask

        if self.norm_first:
            tgt_feature = self._forward_prenorm(tgt_feature, enc_feature,
                                                tgt_san_mask, enc_tgt_mask)
        else:
            tgt_feature = self._forward_postnorm(tgt_feature, enc_feature,
                                                 tgt_san_mask, enc_tgt_mask)

        return tgt_feature

In [ ]:
class TransformerDecoder(nn.Module):
    def __init__(self,
                 decoder_layer,
                 out_channels,
                 num_layers,
                 dim_model,
                 dropout_pe,
                 norm_type_tail,
                 norm_eps,
                 norm_first,
                 add_bias,
                 add_tailnorm,
                 padding_val):
        super().__init__()

        self.emb_layer = nn.Embedding(out_channels,
                                      dim_model,
                                      padding_idx=padding_val)
        self.vocab_size = out_channels

        self.pos_encoder = PositionalEncoding(dim_model, dropout_pe)
        self.layers = nn.ModuleList([copy.deepcopy(decoder_layer) for _ in range(num_layers)])

        # Add LayerNorm at tail position.
        # This is applied only when norm_first is True because
        # post-normalization structure includes tail-normalization in encoder
        # layers.
        if add_tailnorm and norm_first:
            self.norm_tail = create_norm(norm_type_tail, dim_model, norm_eps, add_bias)
        else:
            self.norm_tail = Identity()

        self.head = nn.Linear(dim_model, out_channels)

        self.reset_parameters(dim_model, padding_val)

    def reset_parameters(self, embedding_dim, padding_val):
        # Bellow initialization has strong effect to performance.
        # Please refer.
        # https://github.com/facebookresearch/fairseq/blob/main/fairseq/models/transformer/transformer_base.py#L189
        nn.init.normal_(self.emb_layer.weight, mean=0, std=embedding_dim**-0.5)
        nn.init.constant_(self.emb_layer.weight[padding_val], 0)

        # Please refer.
        # https://github.com/facebookresearch/fairseq/blob/main/fairseq/models/transformer/transformer_decoder.py
        nn.init.xavier_uniform_(self.head.weight)
        nn.init.constant_(self.head.bias, 0.0)

    def forward(self,
                tgt_feature,
                enc_feature,
                tgt_causal_mask,
                enc_tgt_causal_mask,
                tgt_key_padding_mask,
                enc_key_padding_mask):

        tgt_feature = self.emb_layer(tgt_feature) * math.sqrt(self.vocab_size)

        tgt_feature = self.pos_encoder(tgt_feature)
        for layer in self.layers:
            tgt_feature = layer(
                tgt_feature=tgt_feature,
                enc_feature=enc_feature,
                tgt_causal_mask=tgt_causal_mask,
                enc_tgt_causal_mask=enc_tgt_causal_mask,
                tgt_key_padding_mask=tgt_key_padding_mask,
                enc_key_padding_mask=enc_key_padding_mask)
        tgt_feature = apply_norm(self.norm_tail, tgt_feature)

        logit = self.head(tgt_feature)
        return logit

### Transformer CsEMG model

In [ ]:
class TransformerCsEMG(nn.Module):
    def __init__(self,
                 in_channels,
                 inter_channels,
                 out_channels,
                 padding_val,
                 activation="relu",
                 tren_num_layers=1,
                 tren_num_heads=1,
                 tren_dim_ffw=256,
                 tren_dropout_pe=0.1,
                 tren_dropout=0.1,
                 tren_norm_type_sattn="layer",
                 tren_norm_type_ffw="layer",
                 tren_norm_type_tail="layer",
                 tren_norm_eps=1e-5,
                 tren_norm_first=True,
                 tren_add_bias=True,
                 tren_add_tailnorm=True,
                 trde_num_layers=1,
                 trde_num_heads=1,
                 trde_dim_ffw=256,
                 trde_dropout_pe=0.1,
                 trde_dropout=0.1,
                 trde_norm_type_sattn="layer",
                 trde_norm_type_cattn="layer",
                 trde_norm_type_ffw="layer",
                 trde_norm_type_tail="layer",
                 trde_norm_eps=1e-5,
                 trde_norm_first=True,
                 trde_add_bias=True,
                 trde_add_tailnorm=True):
        super().__init__()

        # Feature extraction.
        self.linear = nn.Linear(in_channels, inter_channels)
        self.activation = select_reluwise_activation(activation)

        # Transformer-Encoder.
        enlayer = TransformerEncoderLayer(
            dim_model=inter_channels,
            num_heads=tren_num_heads,
            dim_ffw=tren_dim_ffw,
            dropout=tren_dropout,
            activation=activation,
            norm_type_sattn=tren_norm_type_sattn,
            norm_type_ffw=tren_norm_type_ffw,
            norm_eps=tren_norm_eps,
            norm_first=tren_norm_first,
            add_bias=tren_add_bias)
        self.tr_encoder = TransformerEncoder(
            encoder_layer=enlayer,
            num_layers=tren_num_layers,
            dim_model=inter_channels,
            dropout_pe=tren_dropout_pe,
            norm_type_tail=tren_norm_type_tail,
            norm_eps=tren_norm_eps,
            norm_first=tren_norm_first,
            add_bias=tren_add_bias,
            add_tailnorm=tren_add_tailnorm)

        # Transformer-Decoder.
        delayer = TransformerDecoderLayer(
            dim_model=inter_channels,
            num_heads=trde_num_heads,
            dim_ffw=trde_dim_ffw,
            dropout=trde_dropout,
            activation=activation,
            norm_type_sattn=trde_norm_type_sattn,
            norm_type_cattn=trde_norm_type_cattn,
            norm_type_ffw=trde_norm_type_ffw,
            norm_eps=trde_norm_eps,
            norm_first=trde_norm_first,
            add_bias=trde_add_bias)
        self.tr_decoder = TransformerDecoder(
            decoder_layer=delayer,
            out_channels=out_channels,
            num_layers=trde_num_layers,
            dim_model=inter_channels,
            dropout_pe=trde_dropout_pe,
            norm_type_tail=trde_norm_type_tail,
            norm_eps=trde_norm_eps,
            norm_first=trde_norm_first,
            add_bias=trde_add_bias,
            add_tailnorm=trde_add_tailnorm,
            padding_val=padding_val)

    def forward(self,
                src_feature,
                tgt_feature,
                src_causal_mask,
                src_padding_mask,
                tgt_causal_mask,
                tgt_padding_mask):
        """Forward computation for train.
        """
        # 筋電信号（EMG + STFT）： [B, C, T, F] → C=3チャネル, F=周波数ビン
        N, C, T, F = src_feature.shape
        src_feature = src_feature.permute([0, 2, 1, 3])  # → [B, T, C, F]
        src_feature = src_feature.reshape(N, T, -1)      # → [B, T, C*F]

        src_feature = self.linear(src_feature)

        enc_feature = self.tr_encoder(
            feature=src_feature,
            causal_mask=src_causal_mask,
            src_key_padding_mask=src_padding_mask)

        preds = self.tr_decoder(tgt_feature=tgt_feature,
                                enc_feature=enc_feature,
                                tgt_causal_mask=tgt_causal_mask,
                                enc_tgt_causal_mask=None,
                                tgt_key_padding_mask=tgt_padding_mask,
                                enc_key_padding_mask=src_padding_mask)
        # `[N, T, C]`
        return preds

    def inference(self,
                  src_feature,
                  start_id,
                  end_id,
                  src_padding_mask=None,
                  max_seqlen=62):
        """Forward computation for test.
        """

        # Feature extraction.
        # 筋電信号（EMG + STFT）： [B, C, T, F] → C=3チャネル, F=周波数ビン
        N, C, T, F = src_feature.shape
        src_feature = src_feature.permute([0, 2, 1, 3])  # → [B, T, C, F]
        src_feature = src_feature.reshape(N, T, -1)      # → [B, T, C*F]

        src_feature = self.linear(src_feature)

        enc_feature = self.tr_encoder(
            feature=src_feature,
            causal_mask=None,
            src_key_padding_mask=src_padding_mask)

        # Apply decoder.
        dec_inputs = torch.tensor([start_id]).to(src_feature.device)
        # `[N, T]`
        dec_inputs = dec_inputs.reshape([1, 1])
        preds = None
        pred_ids = [start_id]
        for _ in range(max_seqlen):
            pred = self.tr_decoder(
                tgt_feature=dec_inputs,
                enc_feature=enc_feature,
                tgt_causal_mask=None,
                enc_tgt_causal_mask=None,
                tgt_key_padding_mask=None,
                enc_key_padding_mask=src_padding_mask)
            # Extract last prediction.
            pred = pred[:, -1:, :]
            # `[N, T, C]`
            if preds is None:
                preds = pred
            else:
                # Concatenate last elements.
                preds = torch.cat([preds, pred], dim=1)

            pid = torch.argmax(pred, dim=-1)
            dec_inputs = torch.cat([dec_inputs, pid], dim=-1)

            pid = pid.reshape([1]).detach().cpu().numpy()[0]
            pred_ids.append(int(pid))
            if int(pid) == end_id:
                break

        # `[N, T]`
        pred_ids = np.array([pred_ids])
        return pred_ids, preds

    def inference_beam(self,
                       src_feature,
                       start_id,
                       end_id,
                       src_padding_mask=None,
                       max_seqlen=62,
                       beam_width=5):
        """
        Beam Search デコード + 選ばれたビームの確率列も返す
        """
        N, C, T, F = src_feature.shape
        src_feature = src_feature.permute([0, 2, 1, 3])  # [B, T, C, F]
        src_feature = src_feature.reshape(N, T, -1)      # [B, T, C*F]
        src_feature = self.linear(src_feature)
        enc_feature = self.tr_encoder(
            feature=src_feature,
            causal_mask=None,
            src_key_padding_mask=src_padding_mask)
        # Beam の初期化: [(sequence, log_prob, preds_list)]
        beams = [([start_id], 0.0, [])]
        for _ in range(max_seqlen):
            new_beams = []
            for seq, score, preds_list in beams:
                dec_inputs = torch.tensor([seq]).to(src_feature.device)  # shape [1, T]
                # Decoder forward
                pred = self.tr_decoder(
                    tgt_feature=dec_inputs,
                    enc_feature=enc_feature,
                    tgt_causal_mask=None,
                    enc_tgt_causal_mask=None,
                    tgt_key_padding_mask=None,
                    enc_key_padding_mask=src_padding_mask)
                # 1 ステップ分の logits
                pred_step = pred[:, -1:, :]  # [1, 1, vocab_size]
                log_probs = torch.log_softmax(pred_step, dim=-1)  # [1, 1, vocab_size]
                topk_log_probs, topk_ids = torch.topk(log_probs, beam_width, dim=-1)  # [1, 1, beam_width]
                for i in range(beam_width):
                    next_id = topk_ids[0, 0, i].item()
                    next_log_prob = topk_log_probs[0, 0, i].item()
                    new_seq = seq + [next_id]
                    new_score = score + next_log_prob
                    # preds_list に pred_step を追加
                    new_preds_list = preds_list + [pred_step.squeeze(1)]  # pred_step: [1, 1, vocab_size] → [1, vocab_size]
                    new_beams.append((new_seq, new_score, new_preds_list))
            # スコア順に beam_width に絞る
            new_beams = sorted(new_beams, key=lambda x: x[1], reverse=True)[:beam_width]
            beams = new_beams
            all_eos = all(seq[-1] == end_id for seq, _, _ in beams)
            if all_eos:
                break
        # スコア最大の1本を選ぶ
        best_seq, best_score, best_preds_list = beams[0]
        # preds: [ステップ数, vocab_size]
        best_preds = torch.cat(best_preds_list, dim=0).unsqueeze(0)  # [1, T, vocab_size]
        return np.array([best_seq]), best_preds

In [ ]:
def test_loop_csir_s2s(dataloader,
                       model,
                       device,
                       start_id,
                       end_id,
                       return_pred_times=False,
                       max_seqlen=62,
                       verbose_num=1):
    size = len(dataloader.dataset)
    total_cer = 0
    all_preds = []
    all_refs = []

    # Collect prediction time.
    pred_times = []

    # Switch to evaluation mode.
    model.eval()
    # Main loop.
    print("Start test.")
    start = time.perf_counter()
    with torch.no_grad():
        for batch_idx, batch_sample in enumerate(dataloader):
            feature = batch_sample["feature"]
            tokens = batch_sample["token"]
            tokens_pad_mask = batch_sample["token_pad_mask"]

            check_tokens_format(tokens, tokens_pad_mask, start_id, end_id)

            feature = feature.to(device)
            tokens = tokens.to(device)
            tokens_pad_mask = tokens_pad_mask.to(device)

            frames = feature.shape[-2]

            # Predict.
            pred_start = time.perf_counter()
            pred_ids = inference(model, feature, start_id, end_id, max_seqlen=max_seqlen)
            pred_end = time.perf_counter()
            pred_times.append([frames, pred_end - pred_start])

            # Compute cer.
            # <sos> and <eos> should be removed because they may boost performance.
            # print(tokens)
            # print(pred_ids)
            if batch_idx < verbose_num:
                print("="*40)
                print("Verbose output")
            if batch_idx < verbose_num:
                print(f"Tokens_w_keywords: {tokens}")
                print(f"Preds_w_keywords: {pred_ids}")

            tokens = tokens[tokens_pad_mask]
            if len(tokens.shape) == 2:
                tokens = tokens[0, 1:-1]
            else:
                tokens = tokens[1:-1]
            # pred_ids = pred_ids[0, 1:-1]
            pred_ids = [pid for pid in pred_ids[0] if pid not in [start_id, end_id]]
            if batch_idx < verbose_num:
                print(f"Tokens_wo_keywords: {tokens}")
                print(f"Preds_wo_keywords: {pred_ids}")

            ref_length = len(tokens)
            cer = edit_distance(tokens, pred_ids)
            cer /= ref_length
            total_cer += cer

            # pred_ids は list, tokens は torch.Tensor
            # tokens は tensorなので list に変換
            ref_tokens_list = tokens.cpu().tolist()
            all_refs.append(ref_tokens_list)
            all_preds.append(pred_ids)
            if batch_idx < verbose_num:
                print(f"cer: {cer}")
                print("="*40)

    print(f"Done. Time:{time.perf_counter()-start}")

    # Average cer.
    acer = total_cer / size * 100
    print("Test performance: \n",
          f"Avg cer:{acer:>0.1f}%\n")
    pred_times = np.array(pred_times)
    if return_pred_times:
        retval = (acer, pred_times, all_preds, all_refs)
    else:
        retval = (acer, all_preds, all_refs)
    return retval

# 4. Sanity check

In [ ]:
# Access check.
dataset_dir = Path("dataset_top10")
files = list(dataset_dir.iterdir())
dictionary = [fin for fin in files if ".json" in fin.name][0]
hdf5_files = [fin for fin in files if ".hdf5" in fin.name]

print(dictionary)
print(hdf5_files)

In [ ]:
# Load dictionary.
with open(dictionary, "r") as fread:
    key2token = json.load(fread)

VOCAB = len(key2token)
# Add keywords.
key2token["<sos>"] = VOCAB
key2token["<eos>"] = VOCAB + 1
key2token["<pad>"] = VOCAB + 2
# Reset.
VOCAB = len(key2token)
print(VOCAB)

In [ ]:
import torch.nn.functional as F
# 正規化（例：チャネル毎に0-1正規化）
class NormalizeSpectrogram:
    def __call__(self, sample):
        # sample は dict 形式を想定
        feature = sample["feature"]  # [C, T, F]
        max_vals = feature.max(axis=(1, 2), keepdims=True)
        max_vals[max_vals == 0] = 1
        sample["feature"] = feature / max_vals
        return sample

# 振幅スケーリング（発話の強弱）
class RandomAmplitudeScaling_weak(nn.Module):
    def __init__(self, apply_ratio=1.0, scale_range=[0.9, 1.0]):
        super().__init__()
        self.apply_ratio = apply_ratio
        self.scale_range = scale_range

    def forward(self, sample):
        feature = sample["feature"]  # [C, T, F]
        if torch.rand(1).item() < self.apply_ratio:
            scale_factor = np.random.uniform(*self.scale_range)
            feature = feature * scale_factor
        sample["feature"] = feature
        return sample

class RandomAmplitudeScaling_strong(nn.Module):
    def __init__(self, apply_ratio=1.0, scale_range=[1.0, 1.1]):
        super().__init__()
        self.apply_ratio = apply_ratio
        self.scale_range = scale_range

    def forward(self, sample):
        feature = sample["feature"]  # [C, T, F]
        if torch.rand(1).item() < self.apply_ratio:
            scale_factor = np.random.uniform(*self.scale_range)
            feature = feature * scale_factor
        sample["feature"] = feature
        return sample

class RandomTimeStretch_slow(nn.Module):
    def __init__(self, apply_ratio=1.0, scale_range=[0.9, 1.0]):
        super().__init__()
        self.apply_ratio = apply_ratio
        self.scale_range = scale_range

    def forward(self, sample):
        feature = sample["feature"]  # [C, T, F]
        if isinstance(feature, np.ndarray):
            feature = torch.from_numpy(feature).float()

        if torch.rand(1).item() < self.apply_ratio:
            scale_factor = np.random.uniform(*self.scale_range)
            C, T, feat_dim = feature.shape

            new_T = int(T * scale_factor)

            # [C, T, F] → [C, F, T]
            feature = feature.permute(0, 2, 1)
            feature = feature.reshape(C * feat_dim, T).unsqueeze(0)  # [1, C*F, T]

            feature_stretched = F.interpolate(feature, size=new_T, mode='linear', align_corners=False)

            feature_stretched = feature_stretched.squeeze(0).reshape(C, feat_dim, new_T)
            feature_stretched = feature_stretched.permute(0, 2, 1)  # [C, new_T, F]

            # 元に戻さないでそのまま渡す
            sample["feature"] = feature_stretched

        return sample


class RandomTimeStretch_fast(nn.Module):
    def __init__(self, apply_ratio=1.0, scale_range=[1.0, 1.1]):
        super().__init__()
        self.apply_ratio = apply_ratio
        self.scale_range = scale_range

    def forward(self, sample):
        feature = sample["feature"]  # [C, T, F]
        if isinstance(feature, np.ndarray):
            feature = torch.from_numpy(feature).float()

        if torch.rand(1).item() < self.apply_ratio:
            scale_factor = np.random.uniform(*self.scale_range)
            C, T, feat_dim = feature.shape

            new_T = int(T * scale_factor)

            feature = feature.permute(0, 2, 1)
            feature = feature.reshape(C * feat_dim, T).unsqueeze(0)

            feature_stretched = F.interpolate(feature, size=new_T, mode='linear', align_corners=False)

            feature_stretched = feature_stretched.squeeze(0).reshape(C, feat_dim, new_T)
            feature_stretched = feature_stretched.permute(0, 2, 1)

            # 元に戻さずにそのまま
            sample["feature"] = feature_stretched

        return sample

class RandomMasking(nn.Module):
    def __init__(self, F=8, T=10, num_freq_masks=2, num_time_masks=2, apply_ratio=1.0):
        super().__init__()
        self.F = F  # 周波数マスクパラメータ（最大幅）
        self.T = T  # 時間マスクパラメータ（最大幅）
        self.num_freq_masks = num_freq_masks  # 周波数マスクの適用回数
        self.num_time_masks = num_time_masks  # 時間マスクの適用回数
        self.apply_ratio = apply_ratio

    def forward(self, sample):
        feature = sample["feature"]  # [C, T, F]
        if isinstance(feature, np.ndarray):
            feature = torch.from_numpy(feature).float()

        if torch.rand(1).item() < self.apply_ratio:
            C, t_len, f_len = feature.shape

            # 周波数マスキングを指定回数ループ
            for _ in range(self.num_freq_masks):
                f = np.random.randint(0, self.F)
                f0 = np.random.randint(0, f_len - f) if f_len > f else 0
                feature[:, :, f0:f0+f] = 0

            # 時間マスキングを指定回数ループ
            for _ in range(self.num_time_masks):
                t = np.random.randint(0, self.T)
                t0 = np.random.randint(0, t_len - t) if t_len > t else 0
                feature[:, t0:t0+t, :] = 0

        sample["feature"] = feature
        return sample

In [ ]:
trans_norm = NormalizeSpectrogram()
trans_amp_weak = RandomAmplitudeScaling_weak()
trans_amp_strong = RandomAmplitudeScaling_strong()
trans_time_slow = RandomTimeStretch_slow()
trans_time_fast = RandomTimeStretch_fast()
trans_masking = RandomMasking(F=3, T=3, num_freq_masks=2, num_time_masks=2, apply_ratio=1.0)
trans_insert_token = InsertTokensForS2S(sos_token=key2token["<sos>"], eos_token=key2token["<eos>"])

pre_transforms = Compose([
    trans_insert_token,
    trans_norm,
])

train_transforms = Compose([
    ToTensor()
])

val_transforms = Compose([
    ToTensor()
])

test_transforms = Compose([
    ToTensor()
])
train_transforms_with_mask = Compose([
    ToTensor(),
    trans_masking
])

In [ ]:
# --- データ読み込み ---
def load_samples_from_hdf5(files):
    samples = []
    for fpath in files:
        with h5py.File(fpath, "r") as f:
            for key in f:
                feature = f[key]["feature"][:]
                token = f[key]["token"][:]
                samples.append({"feature": feature, "token": token})
    return samples

train_samples = load_samples_from_hdf5(["dataset_top10/0.hdf5"])

# --- 2. データ拡張 ---
augmented_samples = []
normalized_samples = []

# 正規化した元データを保持
for sample in train_samples:
    sample_norm = deepcopy(sample)
    augmented_samples.append(sample_norm)
    normalized_samples.append(sample_norm)

# 100%をデータ拡張
num_to_augment = int(len(normalized_samples) * 1.0)
random.shuffle(normalized_samples)

#selected_for_amp = normalized_samples[:num_to_augment]
#selected_for_time = normalized_samples[num_to_augment:2*num_to_augment]

selected_for_amp = normalized_samples[:num_to_augment]
selected_for_time = normalized_samples[:num_to_augment]
selected_for_mask = normalized_samples[:num_to_augment]

## 振幅スケーリング
#for sample in selected_for_amp:
#    sample_amp = deepcopy(sample)
#    sample_amp = trans_amp_weak(sample_amp)
#    augmented_samples.append(sample_amp)
#
#for sample in selected_for_amp:
#    sample_amp = deepcopy(sample)
#    sample_amp = trans_amp_strong(sample_amp)
#    augmented_samples.append(sample_amp)
#
## 時間シフト
#for sample in selected_for_time:
#    sample_time = deepcopy(sample)
#    sample_time = trans_time_slow(sample_time)
#    augmented_samples.append(sample_time)
#
#for sample in selected_for_time:
#    sample_time = deepcopy(sample)
#    sample_time = trans_time_fast(sample_time)
#    augmented_samples.append(sample_time)

# --- 3. HDF5として保存 ---
with h5py.File("dataset_top10/3.hdf5", "w") as f:
    for i, sample in enumerate(augmented_samples):
        g = f.create_group(str(i))
        g.create_dataset("feature", data=sample["feature"])
        g.create_dataset("token", data=sample["token"])

In [ ]:
with h5py.File("dataset_top10/3.hdf5", "r") as f:
    sample_count = len(f.keys())  # ファイル内のグループ数をカウント
    print(f"Augmented samples count: {sample_count}")

In [ ]:
# Access check.
dataset_dir = Path("dataset_top10")
files = list(dataset_dir.iterdir())
dictionary = [fin for fin in files if ".json" in fin.name][0]
hdf5_files = [fin for fin in files if ".hdf5" in fin.name]

print(dictionary)
print(hdf5_files)

In [ ]:
batch_size = 32
feature_shape = (3, -1, Sn)
token_shape = (-1,)
merge_fn = partial(merge_padded_batch,
                   feature_shape=feature_shape,
                   token_shape=token_shape,
                   feature_padding_val=0.0,
                   token_padding_val=key2token["<pad>"])

dataset = HDF5Dataset(hdf5_files, pre_transforms=pre_transforms, transforms=train_transforms)

dataloader = DataLoader(dataset, batch_size=batch_size, collate_fn=merge_fn)
try:
    data = next(iter(dataloader))
    feature_origin = data["feature"]
    tokens_origin = data["token"]

    print(feature_origin.shape)
    print(tokens_origin)
except Exception as inst:
    print(inst)

In [ ]:
# Define model.
# in_channels: J * C (130*2=260)
#   J: use_landmarks (130)
#   C: use_channels (2)
# out_channels: 10
in_channels = 3 * Sn  # 例：3チャネル × 60周波数ビン → 180
inter_channels = 64
out_channels = VOCAB
norm_type = "layer"
pad_token = key2token["<pad>"]
model = TransformerCsEMG(
    in_channels=in_channels,
    inter_channels=inter_channels,
    out_channels=out_channels,
    padding_val=pad_token,
    activation="relu",
    tren_num_layers=2,
    tren_num_heads=2,
    tren_dim_ffw=256,
    tren_dropout_pe=0.1,
    tren_dropout=0.1,
    tren_norm_type_sattn=norm_type,
    tren_norm_type_ffw=norm_type,
    tren_norm_type_tail=norm_type,
    tren_norm_eps=1e-5,
    tren_norm_first=True,
    tren_add_bias=True,
    tren_add_tailnorm=True,
    trde_num_layers=2,
    trde_num_heads=2,
    trde_dim_ffw=256,
    trde_dropout_pe=0.1,
    trde_dropout=0.1,
    trde_norm_type_sattn=norm_type,
    trde_norm_type_cattn=norm_type,
    trde_norm_type_ffw=norm_type,
    trde_norm_type_tail=norm_type,
    trde_norm_eps=1e-5,
    trde_norm_first=True,
    trde_add_bias=True,
    trde_add_tailnorm=True)

print(model)

# Sanity check.
sample = next(iter(dataloader))
logit = model(src_feature=sample["feature"],
              tgt_feature=sample["token"],
              src_causal_mask=None,
              src_padding_mask=sample["feature_pad_mask"],
              tgt_causal_mask=make_causal_mask(sample["token_pad_mask"]),
              tgt_padding_mask=sample["token_pad_mask"])
print(logit.shape)

# 5. Train and evaluation

## 5.1 Set common parameters

In [ ]:
# Set common parameters.
batch_size = 32
load_into_ram = True
train_pid = 3
val_pid = 1
test_pid = 2
num_workers = os.cpu_count()
print(f"Using {num_workers} cores for data loading.")
lr = 1e-4
label_smoothing = 0.1
sos_token = key2token["<sos>"]
eos_token = key2token["<eos>"]
pad_token = key2token["<pad>"]
max_seqlen = 20

epochs = 50
eval_every_n_epochs = 1
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using {device} for computation.")

train_hdf5files = [fin for fin in hdf5_files if str(train_pid) in fin.name]
val_hdf5files = [fin for fin in hdf5_files if str(val_pid) in fin.name]

test_hdf5files = [fin for fin in hdf5_files if str(test_pid) in fin.name]

In [ ]:
# Build dataloaders.
#　マスクなし
train_dataset = HDF5Dataset(train_hdf5files,
    pre_transforms=pre_transforms, transforms=train_transforms, load_into_ram=load_into_ram)
#　マスクあり
#train_dataset = HDF5Dataset(train_hdf5files,
#   pre_transforms=pre_transforms, transforms=train_transforms_with_mask, load_into_ram=load_into_ram)
val_dataset = HDF5Dataset(val_hdf5files,
    pre_transforms=pre_transforms, transforms=val_transforms, load_into_ram=load_into_ram)
test_dataset = HDF5Dataset(test_hdf5files,
    pre_transforms=pre_transforms, transforms=test_transforms, load_into_ram=load_into_ram)

train_dataloader = DataLoader(train_dataset, batch_size=batch_size, collate_fn=merge_fn, num_workers=num_workers, shuffle=True)
val_dataloader = DataLoader(val_dataset, batch_size=batch_size, collate_fn=merge_fn, num_workers=num_workers, shuffle=False)
test_dataloader = DataLoader(test_dataset, batch_size=1, collate_fn=merge_fn, num_workers=num_workers, shuffle=False)

In [ ]:
print(len(train_dataset))  # 学習データの総サンプル数
print(len(val_dataset))    # 検証データの総サンプル数
print(len(test_dataset))   # テストデータの総サンプル数

## 5.2 Run training process

In [ ]:
torch.manual_seed(42)  # 再現性のためシードを固定
model_transformer = TransformerCsEMG(
    in_channels=in_channels,
    inter_channels=inter_channels,
    out_channels=out_channels,
    padding_val=pad_token,
    activation="relu",
    tren_num_layers=2,
    tren_num_heads=2,
    tren_dim_ffw=256,
    tren_dropout_pe=0.1,
    tren_dropout=0.1,
    tren_norm_type_sattn=norm_type,
    tren_norm_type_ffw=norm_type,
    tren_norm_type_tail=norm_type,
    tren_norm_eps=1e-5,
    tren_norm_first=True,
    tren_add_bias=True,
    tren_add_tailnorm=True,
    trde_num_layers=2,
    trde_num_heads=2,
    trde_dim_ffw=256,
    trde_dropout_pe=0.1,
    trde_dropout=0.1,
    trde_norm_type_sattn=norm_type,
    trde_norm_type_cattn=norm_type,
    trde_norm_type_ffw=norm_type,
    trde_norm_type_tail=norm_type,
    trde_norm_eps=1e-5,
    trde_norm_first=True,
    trde_add_bias=True,
    trde_add_tailnorm=True)

print(model_transformer)

loss_fn = LabelSmoothingCrossEntropyLoss(
    ignore_indices=pad_token, reduction="mean_temporal_prior",
    label_smoothing=label_smoothing)
optimizer = torch.optim.Adam(model_transformer.parameters(), lr=lr, weight_decay=1e-5)

scheduler = ReduceLROnPlateau(optimizer, patience=3)

In [ ]:
# Train, validation, and evaluation.
model_transformer.to(device)

train_losses = []
val_losses = []
test_cers = []
all_top5_probs = []
print("Start training.")
for epoch in range(epochs):
    print("-" * 80)
    print(f"Epoch {epoch+1}")

    train_loss, train_times = train_loop_csir_s2s(
        train_dataloader, model_transformer, loss_fn, optimizer, device,
        sos_token, eos_token,
        return_pred_times=True)
    val_loss, val_times = val_loop_csir_s2s(
        val_dataloader, model_transformer, loss_fn, device,
        sos_token, eos_token,
        return_pred_times=True)
    val_losses.append(val_loss)

    if (epoch+1) % eval_every_n_epochs == 0:
        cer, test_times, preds, refs = test_loop_csir_s2s(
            test_dataloader, model_transformer, device,
            sos_token, eos_token,
            max_seqlen=max_seqlen,
            return_pred_times=True,
            verbose_num=0)
        test_cers.append(cer)

        # 数値ラベル → 音素（文字）変換用辞書
        # ひらがな1文字の辞書を読み込み
        with open(dataset_dir / "dictionary.json", "r", encoding="utf-8") as f:
            char_to_index = json.load(f)

        # index → ひらがな の変換辞書を作る
        index_to_char = {idx: ch for ch, idx in char_to_index.items()}

        # predsとrefsを「ひらがな文字列」に変換
        preds_str = ["".join([index_to_char[token] for token in seq if token in index_to_char]) for seq in preds]
        refs_str  = ["".join([index_to_char[token] for token in seq if token in index_to_char]) for seq in refs]

        # 出力例：正解と予測が違うサンプルを5つ表示
        shown = 0
        for idx, (pred, ref) in enumerate(zip(preds_str, refs_str)):
            if pred != ref:  # 間違ったサンプルだけ
                shown += 1
                print(f"Sample {shown}")
                print("正解:", ref)
                print("予測:", pred)
                print("正解ID列:", refs[idx])
                print("予測ID列:", preds[idx])
                print("-" * 30)
                if shown >= 5:
                    break

train_losses_trans = np.array(train_losses)
val_losses_trans = np.array(val_losses)
test_cers_trans = np.array(test_cers)

val_losses_trans = np.array(val_losses_trans)
test_cers_trans = np.array(test_cers_trans)
print(f"Minimum validation loss:{val_losses_trans.min()} at {np.argmin(val_losses_trans)+1} epoch.")
print(f"Minimum cer:{test_cers_trans.min()} at {np.argmin(test_cers_trans)*eval_every_n_epochs+1} epoch.")

### Plot result

In [ ]:
import matplotlib.pyplot as plt

plt.grid(axis="y", linestyle="dotted", color="k")

xs = np.arange(1, len(val_losses_trans)+1)
plt.plot(xs, val_losses_trans, label="Transformer", marker=".")
plt.xlabel("Epochs")
plt.ylabel("Loss")
plt.ylim([0.0, 1.5])
plt.legend()
plt.show()

In [ ]:
plt.grid(axis="y", linestyle="dotted", color="k")

xs = np.arange(1, len(test_cers_trans)+1)
plt.plot(xs, test_cers_trans, label="Transformer", marker=".")
plt.xlabel("Epochs")
plt.ylabel("cer")
plt.ylim([0.0, 100.0])
plt.legend()
plt.show()

In [ ]:
num_samples = 3  # 表示したいサンプル数

# --- 変更点: カラースケールを固定値で設定 ---
# データの正規化後のレンジに合わせて適宜数値を調整してください。
# （例: 標準化されているなら -3.0 ~ 3.0 など）
V_MIN = 1.0
V_MAX = 2.2

# 1. バッチを取得
data = next(iter(train_dataloader))

# 2. 可視化の設定
fig, axes = plt.subplots(num_samples * 2, 3, figsize=(15, 4 * num_samples * 2), constrained_layout=True)
fig.suptitle(f'SpecAugment Preview (3 Samples) - Fixed Scale [{V_MIN}, {V_MAX}]', fontsize=16)

for s_idx in range(num_samples):
    # 特定のサンプルの特徴量を取得
    original_feature = data['feature'][s_idx].clone()

    # マスキング適用
    sample = {'feature': original_feature.clone()}
    masked_sample = trans_masking(sample)
    masked_feature = masked_sample['feature']

    # 各チャンネルをループ
    for i in range(3):
        row_orig = s_idx * 2
        row_mask = s_idx * 2 + 1

        # Originalの描画
        ax_orig = axes[row_orig, i]
        im_orig = ax_orig.imshow(original_feature[i].T.numpy(),
                                 aspect='auto', origin='lower',
                                 cmap='viridis', vmin=V_MIN, vmax=V_MAX) # ★固定値を適用
        ax_orig.set_title(f'Sample {s_idx+1} | Orig Ch {i+1}')
        fig.colorbar(im_orig, ax=ax_orig)

        # Maskedの描画
        ax_mask = axes[row_mask, i]
        im_mask = ax_mask.imshow(masked_feature[i].T.numpy(),
                                 aspect='auto', origin='lower',
                                 cmap='viridis', vmin=V_MIN, vmax=V_MAX) # ★固定値を適用
        ax_mask.set_title(f'Sample {s_idx+1} | Masked Ch {i+1}')
        fig.colorbar(im_mask, ax=ax_mask)

plt.show()

In [ ]:
import subprocess
subprocess.run(["apt-get", "install", "-y", "fonts-noto-cjk"], capture_output=True)

# キャッシュを削除
import shutil, os
cache_dir = os.path.expanduser("~/.cache/matplotlib")
if os.path.exists(cache_dir):
    shutil.rmtree(cache_dir)

# ランタイムを再起動せずにフォントを直接指定する方法
from matplotlib import font_manager
font_manager.fontManager.addfont("/usr/share/fonts/opentype/noto/NotoSansCJK-Regular.ttc")

import matplotlib
prop = font_manager.FontProperties(fname="/usr/share/fonts/opentype/noto/NotoSansCJK-Regular.ttc")
matplotlib.rc("font", family=prop.get_name())

In [ ]:
###############################################################
# Attention可視化セル（v2: 型エラー修正版）
# ── 学習後に実行してください ──
###############################################################

import matplotlib.pyplot as plt
import torch
import numpy as np

def plot_attention_weights(model, feature, token_ids, token_to_char,
                           src_padding_mask=None, device="cpu",
                           sample_title="Sample"):
    model.eval()
    model.to(device)
    feature = feature.to(device)
    token_ids = token_ids.to(device)

    with torch.no_grad():
        # ── Encoder へ通す ──────────────────────────────────────
        N, C, T, F = feature.shape
        src = feature.permute([0, 2, 1, 3]).reshape(N, T, -1)
        src = model.linear(src)
        enc_feature = model.tr_encoder(
            feature=src,
            causal_mask=None,
            src_key_padding_mask=src_padding_mask)

        # ── Decoder へ通す（teacher forcing）──────────────────
        tgt_in = token_ids[:, :-1]   # <sos> ... (最後のトークンを除く)

        # ▼ 修正ポイント: dtype=torch.bool を明示する
        tgt_pad_mask = torch.ones(tgt_in.shape, dtype=torch.bool, device=device)
        tgt_causal = make_causal_mask(tgt_pad_mask)

        _ = model.tr_decoder(
            tgt_feature=tgt_in,
            enc_feature=enc_feature,
            tgt_causal_mask=tgt_causal,
            enc_tgt_causal_mask=None,
            tgt_key_padding_mask=tgt_pad_mask,
            enc_key_padding_mask=src_padding_mask)

    # ── Attention重みを取り出す ────────────────────────────────
    num_layers = len(model.tr_decoder.layers)
    num_heads  = model.tr_decoder.layers[0].sattw.shape[1]

    # ── ラベル文字列を作成（<sos>を除いた入力トークン）──────────
    tgt_tokens = token_ids[0, 1:].cpu().tolist()
    tgt_labels = [token_to_char.get(t, str(t)) for t in tgt_tokens]

    # ── 描画 ──────────────────────────────────────────────────
    fig, axes = plt.subplots(
        num_layers * num_heads, 2,
        figsize=(14, 3.5 * num_layers * num_heads),
        constrained_layout=True)
    fig.suptitle(f"Attention weights  [{sample_title}]", fontsize=14)

    # axes が 1D になるケースに対応
    if num_layers * num_heads == 1:
        axes = np.array([[axes[0], axes[1]]])

    row = 0
    for li in range(num_layers):
        layer = model.tr_decoder.layers[li]
        sattw = layer.sattw[0].cpu().numpy()  # [num_heads, Ltgt, Ltgt]
        cattw = layer.cattw[0].cpu().numpy()  # [num_heads, Ltgt, Tsrc]

        for hi in range(num_heads):
            ax_s = axes[row, 0]
            ax_c = axes[row, 1]

            # ── Self-attention ──
            im_s = ax_s.imshow(sattw[hi], aspect="auto", cmap="Blues", vmin=0)
            ax_s.set_title(f"(a) Self-attn  Layer{li}, Head{hi}")
            ax_s.set_xlabel("Key (input token)")
            ax_s.set_ylabel("Query (output token)")
            ax_s.set_xticks(range(len(tgt_labels)))
            ax_s.set_xticklabels(tgt_labels, fontsize=9)
            ax_s.set_yticks(range(len(tgt_labels)))
            ax_s.set_yticklabels(tgt_labels, fontsize=9)
            fig.colorbar(im_s, ax=ax_s, fraction=0.046, pad=0.04)

            # ── Cross-attention ──
            im_c = ax_c.imshow(cattw[hi], aspect="auto", cmap="Blues", vmin=0)
            ax_c.set_title(f"(b) Cross-attn  Layer{li}, Head{hi}")
            ax_c.set_xlabel("Encoder time frame")
            ax_c.set_ylabel("Query (output token)")
            ax_c.set_yticks(range(len(tgt_labels)))
            ax_c.set_yticklabels(tgt_labels, fontsize=9)
            fig.colorbar(im_c, ax=ax_c, fraction=0.046, pad=0.04)

            row += 1

    plt.show()


# ================================================================
# ▼ 実行部分
# ================================================================

token2char = {v: k for k, v in key2token.items()
              if k not in ["<pad>", "<sos>", "<eos>"]}

model_transformer.eval()

sample_batch = next(iter(test_dataloader))

feature_1   = sample_batch["feature"][[0]]
token_ids_1 = sample_batch["token"][[0]]

pad_mask_1 = sample_batch.get("feature_pad_mask", None)
if pad_mask_1 is not None:
    # ▼ 修正ポイント: bool型に変換してからdeviceへ
    pad_mask_1 = pad_mask_1[[0]].bool().to(device)

ref_ids = token_ids_1[0].tolist()
ref_str = "".join([token2char.get(t, "") for t in ref_ids])
print(f"正解: {ref_str}")

plot_attention_weights(
    model=model_transformer,
    feature=feature_1,
    token_ids=token_ids_1,
    token_to_char=token2char,
    src_padding_mask=pad_mask_1,
    device=device,
    sample_title=f"正解: {ref_str}")